In [1]:
import os
os.chdir("../")

In [2]:
import pandas as pd
from src.dsproject.constants import *
from src.dsproject.utils.common import read_yaml, create_directories
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

In [3]:
class ConfigurationManager:
    def __init__(self, config_file_path: Path = CONFIG_FILE_PATH, 
                 params_file_path: Path = PARAMS_FILE_PATH,
                 schema_file_path: Path = SCHEMA_FILE_PATH):
        
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        model_trainer_config = self.config.model_trainer
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        return ModelTrainerConfig(
            root_dir=Path(model_trainer_config.root_dir),
            train_data_path=Path(model_trainer_config.train_data_path),
            test_data_path=Path(model_trainer_config.test_data_path),
            model_name=model_trainer_config.model_name,
            alpha=params.alpha,
            l1_ratio=params.l1_ratio,
            target_column=schema
        )

In [12]:
from src.dsproject import logger
from sklearn.linear_model import ElasticNet
import joblib

class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train_model(self):
        logger.info("Loading training data...")
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        X_train = train_data.drop(columns=[self.config.target_column.name])
        y_train = train_data[self.config.target_column.name]
        X_test = test_data.drop(columns=[self.config.target_column.name])
        y_test = test_data[self.config.target_column.name]

        logger.info("Training the ElasticNet model...")
        model = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio, random_state=42)
        model.fit(X_train, y_train)

        model_path = self.config.root_dir / self.config.model_name
        model_path.parent.mkdir(parents=True, exist_ok=True)
        joblib.dump(model, model_path)
        logger.info(f"Model saved at {model_path}")

In [13]:
try:
    config_manager = ConfigurationManager()
    model_trainer_config = config_manager.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train_model()
except Exception as e:
    logger.exception(f"An error occurred during model training: {e}")

[2026-05-05 13:43:15,092: INFO: common: Reading YAML file from: config\config.yaml]
[2026-05-05 13:43:15,101: INFO: common: Successfully read YAML file: config\config.yaml]
[2026-05-05 13:43:15,104: INFO: common: Reading YAML file from: params.yaml]
[2026-05-05 13:43:15,108: INFO: common: Successfully read YAML file: params.yaml]
[2026-05-05 13:43:15,112: INFO: common: Reading YAML file from: schema.yaml]
[2026-05-05 13:43:15,117: INFO: common: Successfully read YAML file: schema.yaml]
[2026-05-05 13:43:15,121: INFO: common: Creating directory at: artifacts]
[2026-05-05 13:43:15,124: INFO: common: Successfully created directory: artifacts]
[2026-05-05 13:43:15,127: INFO: 1080129352: Loading training data...]
[2026-05-05 13:43:15,148: INFO: 1080129352: Training the ElasticNet model...]
[2026-05-05 13:43:15,167: INFO: 1080129352: Model saved at artifacts\model_trainer\model.joblib]
